# 0b. Profiling de Bronze

Propósito: Evaluar la calidad de cada parquet de bronze con los 8 criterios y generar los reportes en `data/profiling/` (HTML + JSON).

> Requiere haber corrido antes `0_run_bronze.ipynb` (o `python main.py bronze`).

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)

ROOT: c:\Users\user\Downloads\EP-GDM-G6


In [2]:
from pathlib import Path
from app.utils.profiler import profile_directory

reports = profile_directory(Path("data/bronze"), Path("data/profiling"))
print(f"{len(reports)} archivos perfilados -> data/profiling/")

2026-06-19 15:41:42,242 - INFO - Iniciando perfilado de directorio: data\bronze
2026-06-19 15:41:42,255 - INFO - Encontrados 26 archivos parquet para perfilar
2026-06-19 15:41:42,256 - INFO - Procesando archivo 1/26: Ingresos_diccionario.parquet
2026-06-19 15:41:42,256 - INFO - Perfilando archivo: data\bronze\dicts\Ingresos_diccionario.parquet
2026-06-19 15:41:45,986 - INFO - Archivo Ingresos_diccionario.parquet: 63 filas, 3 columnas
2026-06-19 15:41:47,946 - INFO - Archivo Ingresos_diccionario.parquet: Score 100.0% - APROBADO
2026-06-19 15:41:47,950 - INFO - Procesando archivo 2/26: rentas_ano_aplicacion_diccionario.parquet
2026-06-19 15:41:47,951 - INFO - Perfilando archivo: data\bronze\dicts\rentas_ano_aplicacion_diccionario.parquet
2026-06-19 15:41:48,176 - INFO - Archivo rentas_ano_aplicacion_diccionario.parquet: 9 filas, 3 columnas
2026-06-19 15:41:48,792 - INFO - Archivo rentas_ano_aplicacion_diccionario.parquet: Score 100.0% - APROBADO
2026-06-19 15:41:48,795 - INFO - Procesand

26 archivos perfilados -> data/profiling/


In [3]:
# Tabla resumen ordenada por score
import pandas as pd
from pathlib import Path

df = pd.DataFrame([
    {
        "archivo": Path(r["file"]).name,
        "filas": r["rows"],
        "score_%": round(r["overall_score"], 1),
        "estado": "APROBADO" if r["passed"] else "REPROBADO",
    }
    for r in reports
]).sort_values("score_%", ascending=False).reset_index(drop=True)
df

,archivo,filas,score_%,estado
0,Ingresos_diccionario.parquet,63,100.0,APROBADO
1,rentas_ano_aplicacion_diccionario.parquet,9,100.0,APROBADO
2,rentas_esat_estadistica_atm_diccionario.parquet,42,100.0,APROBADO
3,rentas_estadistica_diccionario.parquet,7,100.0,APROBADO
4,rentas_formulario_diccionario.parquet,10,100.0,APROBADO
5,rentas_preguntas_diccionario.parquet,15,100.0,APROBADO
6,rentas_respuestas_diccionario.parquet,11,100.0,APROBADO
7,manifest.parquet,18,100.0,APROBADO
8,SISMEPRE-rentas_respuestas.parquet,250521,100.0,APROBADO
9,RENAMU-984-Modulo1963.parquet,1891,100.0,APROBADO


In [4]:
# Ruta del reporte HTML consolidado
from pathlib import Path
print("Reporte HTML:", Path("data/profiling/profiling_summary.html").resolve())

Reporte HTML: C:\Users\user\Downloads\EP-GDM-G6\data\profiling\profiling_summary.html
